In [18]:
import os
import json
from openai import OpenAI
from google.colab import userdata
from sklearn.metrics.pairwise import cosine_similarity

# **Problem Statement**

Build a minimal ReAct-based AI agent for an IT Helpdesk system.

The goal of the agent is to help employees troubleshoot common IT issues by:

* reasoning about the user query,
* deciding when to use a tool,
* retrieving relevant troubleshooting information,
* and responding only using verified tool outputs.

In [53]:
client = OpenAI(api_key=userdata.get('openai_IK')) # for colab

In [44]:
## For non-colab use

# import getpass

# OPENAI_API_KEY = getpass.getpass('OpenAI API Key:')
# client = OpenAI(api_key=OPENAI_API_KEY)

OpenAI API Key:··········


In [45]:
def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )

    return response.data[0].embedding

In [46]:
# Mimicing the Vector DB

issue_embeddings = {}

list_of_operations = {
  "vpn not connecting": "Restart VPN client and check internet connection",
  "microphone not working": "Verify microphone permissions in Teams settings",
  "laptop overheating": "Restart the laptop and make sure the laptop fan is not covered"
}

for issue in list_of_operations.keys():
    issue_embeddings[issue] = get_embedding(issue)

In [47]:
def search_query(query: str):

  print(f"-> TOOL: Fetching solution against the query ...")

  query_embedding = get_embedding(query)
  best_score = -1
  issue_found = ""

  for issue, embedding in issue_embeddings.items():
    score = cosine_similarity([embedding], [query_embedding])[0][0]

    if score > best_score:
      best_score = score
      issue_found = issue

  if issue_found == "":
    return json.dumps({query :"No issue found"})

  return json.dumps({issue_found :list_of_operations[issue_found]})

In [48]:
def escalate_to_customer_support(summary: str):

  print(f"-> TOOL: Creating Support ticket ..")

  summary = json.loads(summary)
  query = summary["search_query"]

  return json.dumps({query: "Ticket raised to the customer support", "status": "Successful"})

In [49]:
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "search_query",
            "description": "Checks the for the closet query and providing the solution based on that",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Customer's problem statement"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "escalate_to_customer_support",
            "description": "If no solution present in the db, it creates the customer support ticket",
            "parameters": {
                "type": "object",
                "properties": {
                    "summary": {"type": "string", "description": "A JSON string containing the Customer's problem statement and the instructions against the query"}
                },
                "required": ["summary"]
            }
        }
    }
]

In [50]:
# Map functions for the agent execution loop
AVAILABLE_FUNCTIONS = {
    "search_query": search_query,
    "escalate_to_customer_support": escalate_to_customer_support
}

Agent Is ready to work

In [51]:
def run_it_agent(user_issue: str):
    print(f"\n--- New Incident: {user_issue} ---")

    messages = [
        {"role": "system", "content": "You are a Level 1 IT Responder. Investigate the customer issue. "
                                      "The customer's issue solution is present in the table, but if not found then raise the customer support ticket"},
        {"role": "user", "content": user_issue}
    ]

    collected_data = {}

    while True:
        print("\n[AI Thinking...]")
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=tools_schema,
            tool_choice="auto"
        )

        response_message = response.choices[0].message

        messages.append(response_message)

        if response_message.tool_calls:
          for tool in response_message.tool_calls:
            func_name = tool.function.name
            func_args = json.loads(tool.function.arguments)

            # Retrieve the actual python function based on name
            function_to_call = AVAILABLE_FUNCTIONS.get(func_name)

            if function_to_call:

              if func_name == 'search_query':
                function_response = function_to_call(**func_args)
                collected_data[func_name] = function_response
              elif func_name == 'escalate_to_customer_support':
                func_args = {"summary": json.dumps(collected_data)}
                function_response = function_to_call(**func_args)

              messages.append({
                  "role": "tool",
                  "tool_call_id": tool.id,
                  "name": func_name,
                  "content": function_response
              })

        else:
          print(f"\n[FINAL RESPONSE]: {response_message.content}")
          break


**Test the Agent**

In [52]:
run_it_agent('mic is not working')


--- New Incident: mic is not working ---

[AI Thinking...]
-> TOOL: Fetching solution against the query ...

[AI Thinking...]

[FINAL RESPONSE]: Please verify the microphone permissions in your application or device settings. If you're using an application like Microsoft Teams, ensure that the microphone permission is enabled within the app settings. Let me know if that resolves your issue!


## Agent on Duty

In [41]:
while True:

  print("Press q or exit to stop")

  user_issue = input("Enter the customer issue: ")

  if user_issue == 'q' or user_issue.lower() == "exit":
    break

  run_it_agent(user_issue)

Press q or exit to stop
Enter the customer issue: vpn issue

--- New Incident: vpn issue ---

[AI Thinking...]
-> TOOL: Fetching solution against the query ...

[AI Thinking...]

[FINAL RESPONSE]: To resolve your VPN issue, please try restarting your VPN client and checking your internet connection. If the problem persists, feel free to ask for additional assistance.
Press q or exit to stop
Enter the customer issue: phone issue

--- New Incident: phone issue ---

[AI Thinking...]
-> TOOL: Fetching solution against the query ...

[AI Thinking...]

[FINAL RESPONSE]: Here is a solution for your issue:

- **Microphone not working**: Verify microphone permissions in Teams settings.

If this solution doesn't resolve your issue, let me know so I can further assist you!
Press q or exit to stop
Enter the customer issue: mobile phone issue

--- New Incident: mobile phone issue ---

[AI Thinking...]
-> TOOL: Fetching solution against the query ...

[AI Thinking...]

[FINAL RESPONSE]: It seems lik